<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>

<h1>Breakout Exercise: RAG on the History of Saudi Arabia</h1>

Built on the WeCloudData "Simple RAG with LangChain and FAISS" demo.

---

## Exercise checklist

| # | Requirement | Where it happens |
|---|---|---|
| 1 | Choose a topic | **History of Saudi Arabia** |
| 2 | Pick 4–5 websites | Section 1 — 5 sources |
| 3 | Load the websites | Section 1 — `WebBaseLoader` |
| 4 | Chunk them | Section 2 — `RecursiveCharacterTextSplitter` |
| 5 | Vectorize | Section 3 — `OpenAIEmbeddings` + `FAISS` |
| 6 | Load top 6 documents | Section 4 — `search_kwargs={"k": 6}` |
| 7 | Show 3 different usages | Sections 6, 7, 8 |

## The pipeline

**Index system:** load → chunk → embed → store
**Retrieval system:** user query → embed query → vector search → return top 6 chunks
**Augment system:** stuff those 6 chunks into the prompt → LLM answers with sources


In [ ]:
!pip install -q langchain langchain-classic langchain-community langchain-openai
!pip install -q faiss-cpu tiktoken openai beautifulsoup4

In [ ]:
import os

# WebBaseLoader warns if no user agent is set, so set it before loading anything.
os.environ["USER_AGENT"] = "WCD-RAG-Exercise/1.0 (classroom demo)"

# --- OpenAI key ---
try:
    from google.colab import userdata          # running in Colab
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:                              # running locally
    if not os.environ.get("OPENAI_API_KEY"):
        from getpass import getpass
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

# 1. Index System — Load documents

**Topic: History of Saudi Arabia.** Five sources, chosen so they cover different eras
instead of repeating the same overview:

| Source | Covers |
|---|---|
| History of Saudi Arabia | Prehistory → modern kingdom, the umbrella article |
| First Saudi state | Diriyah, 1727/1744 – 1818 |
| Second Saudi state | Turki bin Abdullah, 1824 – 1891 |
| Unification of Saudi Arabia | The 1902–1932 campaign |
| Ibn Saud | The founder's biography |

Coverage matters more than count here — five copies of the same page would give the
retriever six near-identical chunks and the answers would get worse, not better.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer

# --- 4-5 websites about our topic ---
SOURCES = {
    "History of Saudi Arabia": "https://en.wikipedia.org/wiki/History_of_Saudi_Arabia",
    "First Saudi State":       "https://en.wikipedia.org/wiki/First_Saudi_state",
    "Second Saudi State":      "https://en.wikipedia.org/wiki/Second_Saudi_state",
    "Unification (1902-1932)": "https://en.wikipedia.org/wiki/Unification_of_Saudi_Arabia",
    "Ibn Saud (founder)":      "https://en.wikipedia.org/wiki/Ibn_Saud",
}

# Wikipedia puts the real article inside <div class="mw-parser-output">.
# Straining to that div drops menus, footers and sidebars, so we embed article
# text instead of navigation noise.
article_only = {"parse_only": SoupStrainer("div", {"class": "mw-parser-output"})}

docs = {}
for name, url in SOURCES.items():
    loader = WebBaseLoader(url, bs_kwargs=article_only)
    loaded = loader.load()

    # Fallback: if the strainer matched nothing (non-Wikipedia site), load the whole page.
    if not loaded or len(loaded[0].page_content.strip()) < 500:
        loaded = WebBaseLoader(url).load()

    for d in loaded:
        d.metadata["source_name"] = name      # friendlier than a raw URL when we cite later
    docs[name] = loaded
    print(f"{name:<26} {len(loaded)} document(s), {len(loaded[0].page_content):>7,} characters")

In [ ]:
# Peek at one document to confirm we actually got history text and not a cookie banner
sample = docs["First Saudi State"][0]
print("METADATA:", sample.metadata)
print("-" * 70)
print(sample.page_content[:800])

### Chunk documents

Why chunk at all? Two reasons:
1. An embedding of a 60,000-character article is an average of everything in it, so it
   matches nothing precisely.
2. We stuff the retrieved text into the prompt — whole articles would blow the context window.

`chunk_size=800` with `chunk_overlap=120` works well for encyclopedic prose: big enough to
hold a full paragraph of narrative, and the overlap keeps a sentence that straddles a
boundary from losing its context.

More about text splitters [here](https://python.langchain.com/docs/concepts/text_splitters/).

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    length_function=len,
)

chunks = {name: text_splitter.split_documents(d) for name, d in docs.items()}

total = 0
for name, ch in chunks.items():
    print(f"Number of chunks for {name:<26} {len(ch)}")
    total += len(ch)
print("-" * 50)
print(f"TOTAL CHUNKS: {total}")

In [ ]:
# What a single chunk actually looks like
print(chunks["Unification (1902-1932)"][10].page_content)

# 2. Vectorize — embed and store in FAISS

`CacheBackedEmbeddings` writes each embedding to `./cache/`, keyed by a hash of the text.
Re-running this notebook then costs nothing for text that hasn't changed — which matters
here because five Wikipedia articles is a few hundred chunks.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import time

try:                                                   # LangChain 1.x
    from langchain_classic.embeddings import CacheBackedEmbeddings
    from langchain_classic.storage import LocalFileStore
except ImportError:                                    # older LangChain
    from langchain.embeddings import CacheBackedEmbeddings
    from langchain.storage import LocalFileStore

# 1. Local cache store
store_path = "./cache/"
os.makedirs(store_path, exist_ok=True)
store = LocalFileStore(store_path)
print(f"Cache store initialized at {store_path}")

# 2. Core embedding model
core_embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 3. Cache-backed wrapper
embedder = CacheBackedEmbeddings.from_bytes_store(
    core_embeddings_model,
    store,
    namespace=core_embeddings_model.model,
)
print("Embedder configured to use the cache.")

In [ ]:
names = list(chunks.keys())

# Build the index from the first source...
first = names[0]
print(f"Creating FAISS index from '{first}' ({len(chunks[first])} chunks)...")
start = time.time()
vectorstore = FAISS.from_documents(chunks[first], embedder)
print(f"  done in {time.time() - start:.2f}s -> {vectorstore.index.ntotal} vectors")

# ...then add the rest.
for name in names[1:]:
    start = time.time()
    vectorstore.add_documents(chunks[name])
    print(f"Added '{name}' ({len(chunks[name])} chunks) in {time.time() - start:.2f}s "
          f"-> {vectorstore.index.ntotal} vectors total")

print("-" * 60)
print(f"Vector store holds {vectorstore.index.ntotal} vectors of "
      f"{vectorstore.index.d} dimensions each.")

In [ ]:
# Proof that the cache works: re-adding the same chunks skips the API entirely
start = time.time()
vectorstore.add_documents(chunks[first])
print(f"Re-adding cached chunks took {time.time() - start:.4f}s (no embedding API calls)")

# Undo that duplicate so the index stays clean
vectorstore = FAISS.from_documents(chunks[first], embedder)
for name in names[1:]:
    vectorstore.add_documents(chunks[name])
print(f"Index rebuilt clean: {vectorstore.index.ntotal} vectors")

# 3. Retrieval System — load the top 6 documents

This is the exercise requirement: **top 6 documents**. The demo used
`vectorstore.as_retriever()`, which silently defaults to k=4, so we pass `k` explicitly.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})
print("Retriever ready — returns top 6 chunks per query.")

In [ ]:
# Retrieval on its own, before any LLM is involved.
# Lower distance score = closer match.
query = "Why did the First Saudi State fall?"

results = vectorstore.similarity_search_with_score(query, k=6)

print(f"QUERY: {query}\n" + "=" * 78)
for i, (doc, score) in enumerate(results, 1):
    print(f"\n[{i}] {doc.metadata['source_name']}  (distance {score:.4f})")
    print(doc.page_content[:250].replace("\n", " ") + "...")

# 4. Augment System — LLM + prompt

A custom prompt is added on top of the demo. The default RetrievalQA prompt lets the model
fall back on what it already knows about Saudi history, which defeats the point of RAG —
we wouldn't be able to tell retrieved facts from memorized ones. This prompt forces it to
answer from the retrieved chunks and to say so when they don't cover the question.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

try:
    from langchain_classic.chains import RetrievalQA
    from langchain_classic.callbacks import StdOutCallbackHandler
except ImportError:
    from langchain.chains import RetrievalQA
    from langchain.callbacks import StdOutCallbackHandler

handler = StdOutCallbackHandler()

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0,
    streaming=False,
)

PROMPT = PromptTemplate(
    template="""You are a history assistant answering questions about the history of Saudi Arabia.
Use ONLY the context below. If the context does not contain the answer, say
"The retrieved sources do not cover this" instead of guessing.
Mention dates and names precisely when the context provides them.

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"],
)

qa_with_sources_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,                      # k = 6
    return_source_documents=True,
    chain_type_kwargs={"prompt": PROMPT},
)
print("RAG chain ready.")

In [ ]:
def ask(question, show_chunks=False):
    """Run the full RAG pipeline and print the answer plus which sources fed it."""
    response = qa_with_sources_chain.invoke({"query": question})

    print("QUESTION:", question)
    print("=" * 78)
    print(response["result"])
    print("\n" + "-" * 78)
    print(f"Answered from {len(response['source_documents'])} retrieved chunks:")
    for name in dict.fromkeys(d.metadata["source_name"] for d in response["source_documents"]):
        print("  -", name)

    if show_chunks:
        print("\nRETRIEVED TEXT")
        for i, d in enumerate(response["source_documents"], 1):
            print(f"\n[{i}] {d.metadata['source_name']}")
            print(d.page_content[:300].replace("\n", " ") + "...")

    return response

# Usage 1 — Factual lookup

The simplest RAG job: a specific fact that lives in one or two chunks. Watch which
sources come back — this should pull mostly from the *First Saudi State* article.

In [ ]:
response_1 = ask("When was the First Saudi State founded, who founded it, "
                 "and what city was its capital?", show_chunks=True)

# Usage 2 — Synthesis across multiple sources

Harder, and the real argument for a vector store over a keyword search. No single article
answers this well, so the six retrieved chunks should come from *different* sources and the
LLM has to combine them.

In [ ]:
response_2 = ask("Compare the three Saudi states: how did each one begin, "
                 "and how did the first two come to an end?")

# Usage 3 — Content generation from retrieved facts

RAG isn't only for Q&A. Here the LLM writes something new, but every date in it has to be
grounded in the retrieved chunks. This is the same shape as "write a tour brochure" in the
tourism version of the exercise.

In [ ]:
response_3 = ask("Write a short chronological timeline of the unification of Saudi Arabia "
                 "from 1902 to 1932, one line per event, with the year in front.")

In [ ]:
# Full raw response object, so you can see what the chain returns
response_3.keys(), len(response_3["source_documents"])

### Bonus — what happens when the sources don't cover the question

Worth showing in the demo. It proves the grounding in the prompt actually holds, instead of
the model quietly inventing an answer.

In [ ]:
_ = ask("What is the ticket price to visit At-Turaif in Diriyah today?")

# Save the index (optional)

Rebuilding costs API calls. Saving lets you reload the vector store in a later session.

In [ ]:
vectorstore.save_local("saudi_history_faiss")
print("Saved to ./saudi_history_faiss")

# To reload later:
# vectorstore = FAISS.load_local(
#     "saudi_history_faiss", embedder, allow_dangerous_deserialization=True
# )

---

## Talking points for the presentation

- **Why these five sources:** each covers a different era, so the six retrieved chunks carry
  six different facts rather than six paraphrases of one.
- **Why `mw-parser-output`:** without straining, roughly a third of each Wikipedia page is
  navigation and reference boilerplate, and those chunks get embedded and can be retrieved.
- **Why k=6:** more chunks means more recall but a longer prompt and more distraction. At
  k=6 with 800-character chunks we hand the model about 4,800 characters of context.
- **The distance scores in Section 3** show retrieval quality directly — if the top score is
  poor, the problem is chunking or sources, not the LLM.
- **If you swap in a non-Wikipedia site,** check the character count printed at load time.
  Many government and tourism sites render with JavaScript, and `WebBaseLoader` will return
  an almost empty page. `SeleniumURLLoader` or `PlaywrightURLLoader` handles those.
